# 11 — Deployment Validation: US Project 05 Final Portfolio

Thin orchestrator. Validates an **already-built deployment candidate** — the
Project 05 final US portfolio (`weighted_multi_strategy_with_strategy_and_portfolio_smooth_dd`).
It loads the saved return series and daily weight panel, then hands them to
`run_deployment_validation`. No strategy is reconstructed and
`run_market_robustness` is not used here.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

/Users/rawls/quant-lab


In [2]:
import pandas as pd

from src.analysis.deployment import (
    run_deployment_validation,
    select_best_per_group,
    pivot_metric_table,
)
from src.analysis.turnover import summarize_turnover

## 1. Load the deployment candidate

Two saved artifacts from `exp_005_risk_engine_final/`:

* `final_weighted_multi_strategy_portfolio_dd.csv` — the portfolio return /
  equity series (after both strategy- and portfolio-level drawdown overlays).
* `final_daily_weights.csv` — the daily asset-level weight panel
  (`Date, Ticker, Weight`) carrying the real trading dates.


In [3]:
EXP_DIR = PROJECT_ROOT / "experiments/completed/exp_005_risk_engine_final"

ret_df = pd.read_csv(EXP_DIR / "final_weighted_multi_strategy_portfolio_dd.csv")
weights_long = pd.read_csv(EXP_DIR / "final_daily_weights.csv", parse_dates=["Date"])

print("return series:", ret_df.shape, list(ret_df.columns))
print("weight panel :", weights_long.shape, list(weights_long.columns))

return series: (2558, 3) ['portfolio_return', 'equity_curve', 'portfolio_dd_exposure']
weight panel : (33254, 3) ['Date', 'Ticker', 'Weight']


## 2. Build `returns`, `weights`, `equity`

The return CSV is positional (no date column); the weight panel carries the
dates. Both have the same 2,558 trading days in the same order, so the sorted
weight-panel dates index the return and equity series.


In [4]:
# Wide weight matrix: dates x tickers
weights = (
    weights_long
    .pivot(index="Date", columns="Ticker", values="Weight")
    .sort_index()
    .fillna(0.0)
)

# Attach the dated index to the positional return / equity series
dates = weights.index
assert len(dates) == len(ret_df), (len(dates), len(ret_df))

returns = pd.Series(ret_df["portfolio_return"].to_numpy(), index=dates, name="returns")
equity = pd.Series(ret_df["equity_curve"].to_numpy(), index=dates, name="equity")

print("returns:", returns.shape, "| weights:", weights.shape, "| equity:", equity.shape)
print("date range:", dates.min().date(), "->", dates.max().date())
returns.head()

returns: (2558,) | weights: (2558, 20) | equity: (2558,)
date range: 2016-01-04 -> 2026-03-06


Date
2016-01-04    0.002917
2016-01-05    0.005973
2016-01-06   -0.002303
2016-01-07   -0.000840
2016-01-08   -0.008182
Name: returns, dtype: float64

## 3. Run the deployment-validation battery


In [5]:
TRANSACTION_COSTS = [0, 2, 5, 10, 20, 50]
REBALANCE_FREQUENCIES = [1, 2, 5, 10, "weekly"]

result = run_deployment_validation(
    returns=returns,
    weights=weights,
    equity=equity,
    transaction_costs=TRANSACTION_COSTS,
    rebalance_frequencies=REBALANCE_FREQUENCIES,
)

sorted(result.keys())

['rebalance_analysis',
 'rebalance_cost_grid',
 'regime_analysis',
 'rolling_metrics',
 'transaction_cost_stress',
 'turnover']

## 4. Turnover summary


In [6]:
turnover_summary = pd.Series(summarize_turnover(result["turnover"]), name="Turnover")
turnover_summary.to_frame()

,Turnover
mean,0.120094
median,0.100171
max,0.560404
p95,0.284440


## 5. Transaction-cost stress (as-deployed weight path)


In [7]:
cost_stress = result["transaction_cost_stress"]
cost_stress

,Cost bps,Sharpe,MDD,CAGR,Calmar,Mean Turnover
0,0,2.090113,-0.366949,0.377021,1.027446,0.120094
1,2,2.052352,-0.368698,0.368726,1.000074,0.120094
2,5,1.995681,-0.371312,0.356376,0.959773,0.120094
3,10,1.901156,-0.375646,0.336038,0.894561,0.120094
4,20,1.711843,-0.384223,0.296268,0.771083,0.120094
5,50,1.142168,-0.409260,0.183882,0.449303,0.120094


## 6. Rebalance analysis (turnover by cadence)


In [8]:
rebalance = result["rebalance_analysis"]
rebalance

,Rebalance Frequency,Mean Turnover,Median Turnover,Max Turnover,P95 Turnover
0,1,0.120094,0.100171,0.560404,0.284440
1,2,0.080067,0.000000,0.629559,0.301436
2,5,0.045911,0.000000,0.632745,0.302736
3,10,0.028381,0.000000,0.683104,0.261718
4,weekly,0.046753,0.000000,0.694892,0.302940


## 7. Best rebalance frequency by cost

From the `(rebalance frequency x cost)` grid, pick the frequency with the
highest net Sharpe at each cost level.


In [9]:
grid = result["rebalance_cost_grid"]

best_rebalance_by_cost = select_best_per_group(
    grid, group_cols=["Cost bps"], score_col="Sharpe", maximize=True
)
best_rebalance_by_cost[["Cost bps", "Rebalance Frequency", "Sharpe", "CAGR", "MDD", "Mean Turnover"]]

,Cost bps,Rebalance Frequency,Sharpe,CAGR,MDD,Mean Turnover
0,0,1,2.090113,0.377021,-0.366949,0.120094
1,2,10,2.081099,0.375054,-0.367448,0.028381
2,5,10,2.067565,0.372110,-0.368196,0.028381
3,10,10,2.044972,0.367216,-0.369440,0.028381
4,20,10,1.999660,0.357477,-0.371922,0.028381
5,50,10,1.862818,0.328654,-0.379311,0.028381


## 8. Deployment decision pivot

Net Sharpe across the full `rebalance frequency x cost` grid — the decision
matrix for choosing a deployment cadence under a given cost assumption.


In [10]:
decision_pivot = pivot_metric_table(
    grid, index="Rebalance Frequency", columns="Cost bps", value_col="Sharpe"
)
decision_pivot

Cost bps,0,2,5,10,20,50
Rebalance Frequency,,,,,,
1,2.090113,2.052352,1.995681,1.901156,1.711843,1.142168
10,2.090113,2.081099,2.067565,2.044972,1.999660,1.862818
2,2.090113,2.064904,2.027068,1.963945,1.837485,1.456804
5,2.090113,2.075594,2.053799,2.017426,1.944513,1.724681
weekly,2.090113,2.075323,2.053119,2.016067,1.941800,1.717922


## 9. Save US-specific outputs


In [11]:
results_dir = PROJECT_ROOT / "research/project_06_deployment_validation/results"
results_dir.mkdir(exist_ok=True)

turnover_summary.to_frame().to_csv(results_dir / "us_deployment_turnover_summary.csv")
cost_stress.to_csv(results_dir / "us_deployment_cost_stress.csv", index=False)
rebalance.to_csv(results_dir / "us_deployment_rebalance.csv", index=False)
best_rebalance_by_cost.to_csv(results_dir / "us_deployment_best_rebalance_by_cost.csv", index=False)
decision_pivot.to_csv(results_dir / "us_deployment_decision_pivot.csv")

print("Saved US deployment-validation outputs to", results_dir)

Saved US deployment-validation outputs to /Users/rawls/quant-lab/research/project_06_deployment_validation/results
